<a href="https://colab.research.google.com/github/mlbyhoang/marketing-attribution/blob/main/01_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 – Data Preparation

This notebook prepares customer journey data for the marketing attribution project.

The raw data contains advertising interactions and CM360 activities.
The goal of this notebook is to clean and structure the data for:

1. Conversion prediction
2. Multi-touch attribution

In [1]:
# Load ibraries
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Connect to Google Drive
from google.colab import drive, userdata
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Load data
DATA_PATH = userdata.get("MTA_DATA_PATH")

df = pd.read_csv(DATA_PATH, delimiter=",")
df.head()

,conversion_id,activity_time,activity,impression_count,click_count,interaction_time,interaction_number,campaign,interaction_type,placement,floodlight_variable_dimension21,activity_count
0,3800910923227433881,2026-02-11 17:55:06,ProductView_Server,0,3,2026-02-11 17:43:10,3,DART Search,Click,deinhandy.de GoogleAds Search,GA1.1.1833507673.1768199735,0
1,5459534759657986833,2026-02-20 10:11:46,Pageview_Server,0,3,2026-02-20 10:11:42,1,DART Search,Click,deinhandy.de GoogleAds Search,GA1.1.1682335648.1771578682,0
2,7173993360599741160,2026-02-10 21:05:52,Pageview_Server,0,3,2026-02-10 21:05:28,1,DART Search,Click,deinhandy.de GoogleAds Search,GA1.1.870244807.1769543999,0
3,7658577527247096958,2026-02-17 20:38:49,Pageview_Server,0,3,2026-02-17 14:22:36,2,DART Search,Click,deinhandy.de GoogleAds Search,GA1.1.239276648.1771334558,0
4,7729130632504957835,2026-02-16 09:46:32,Pageview_Server,0,3,2026-02-16 09:11:59,2,DART Search,Click,deinhandy.de GoogleAds Search,GA1.1.825061615.1769627264,0


In [4]:
# Show data types and columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11174533 entries, 0 to 11174532
Data columns (total 12 columns):
 #   Column                           Dtype 
---  ------                           ----- 
 0   conversion_id                    uint64
 1   activity_time                    object
 2   activity                         object
 3   impression_count                 int64 
 4   click_count                      int64 
 5   interaction_time                 object
 6   interaction_number               int64 
 7   campaign                         object
 8   interaction_type                 object
 9   placement                        object
 10  floodlight_variable_dimension21  object
 11  activity_count                   int64 
dtypes: int64(4), object(7), uint64(1)
memory usage: 1023.1+ MB


In [5]:
# Show rows and columns
df.shape

(11174533, 12)

In [6]:
# Rename columns
df.rename(columns={"floodlight_variable_dimension21": "ga_cid"}, inplace=True)
df.head()

,conversion_id,activity_time,activity,impression_count,click_count,interaction_time,interaction_number,campaign,interaction_type,placement,ga_cid,activity_count
0,3800910923227433881,2026-02-11 17:55:06,ProductView_Server,0,3,2026-02-11 17:43:10,3,DART Search,Click,deinhandy.de GoogleAds Search,GA1.1.1833507673.1768199735,0
1,5459534759657986833,2026-02-20 10:11:46,Pageview_Server,0,3,2026-02-20 10:11:42,1,DART Search,Click,deinhandy.de GoogleAds Search,GA1.1.1682335648.1771578682,0
2,7173993360599741160,2026-02-10 21:05:52,Pageview_Server,0,3,2026-02-10 21:05:28,1,DART Search,Click,deinhandy.de GoogleAds Search,GA1.1.870244807.1769543999,0
3,7658577527247096958,2026-02-17 20:38:49,Pageview_Server,0,3,2026-02-17 14:22:36,2,DART Search,Click,deinhandy.de GoogleAds Search,GA1.1.239276648.1771334558,0
4,7729130632504957835,2026-02-16 09:46:32,Pageview_Server,0,3,2026-02-16 09:11:59,2,DART Search,Click,deinhandy.de GoogleAds Search,GA1.1.825061615.1769627264,0


In [7]:
# Drop irrelevant columns
df2 = df.drop(columns=["conversion_id", "impression_count", "click_count", "placement", "activity_count"])
df2.head()

,activity_time,activity,interaction_time,interaction_number,campaign,interaction_type,ga_cid
0,2026-02-11 17:55:06,ProductView_Server,2026-02-11 17:43:10,3,DART Search,Click,GA1.1.1833507673.1768199735
1,2026-02-20 10:11:46,Pageview_Server,2026-02-20 10:11:42,1,DART Search,Click,GA1.1.1682335648.1771578682
2,2026-02-10 21:05:52,Pageview_Server,2026-02-10 21:05:28,1,DART Search,Click,GA1.1.870244807.1769543999
3,2026-02-17 20:38:49,Pageview_Server,2026-02-17 14:22:36,2,DART Search,Click,GA1.1.239276648.1771334558
4,2026-02-16 09:46:32,Pageview_Server,2026-02-16 09:11:59,2,DART Search,Click,GA1.1.825061615.1769627264


In [8]:
# Convert time features to datetime
df2["interaction_time"] = pd.to_datetime(df2["interaction_time"], format="%Y-%m-%d %H:%M:%S")
df2["activity_time"] = pd.to_datetime(df2["activity_time"], format="%Y-%m-%d %H:%M:%S")

In [9]:
# Replace values in 'channel' column
df2["channel"] = df2["campaign"].replace({
    "DART Search": "paid_search",
    "Natural Search": "organic_search",
    "Deinhandy_DE_Alwayson_Affiliate": "dh_affiliate",
    "Sparhandy_DE_Alwayson_DV360_Display_2024": "sh_display",
    "Sparhandy_DE_Alwayson_Affiliate": "sh_affiliate",
    "Sparhandy_DE_Alwayson_DV360_Native_2024": "sh_native",
    "Sparhandy_DE_Alwayson_DV360_DemandGen_2025": "sh_display",
    "Sparhandy_DE_Google-Push_Twitch": "sh_twitch",
    "Deinhandy_DE_Alwayson_DV360_Display_2024": "dh_display",
    "High_DE_Alwayson_DV360_Display_2024": "high_display",
    "Sparhandy_DE_Alwayson_E-Mail": "sh_e_mail",
    "sparsim_DE_Alwayson_DV360_Display": "sparsim_display",
    "simyo_DE_Alwayson_DV360_Display": "simyo_display",
    "DeineSIM_DE_Alwayson_DV360_Display": "deinesim_display",
    "DoubleClick Social: facebook 10152990684218857 #2": "sh_paid_facebook",
    "DoubleClick Social: facebook 0": "dh_paid_facebook",
    "DoubleClick Social: facebook 3418117581757226": "high_paid_facebook",
    "DoubleClick Social: facebook 617516985393525": "dh_paid_facebook",
    "Deinhandy_DE_Alwayson_E-Mail": "dh_e_mail",
    "simyo_DE_Alwayson_Affiliate": "simyo_affiliate",
    "sparsim_DE_Alwayson_DV360_Native": "sparsim_native",
    "Sparhandy_DE_Alwayson_Facebook_Organic": "sh_organic_facebook",
    "Deinhandy_DE_Alwayson_Facebook_Organic": "dh_organic_facebook",
    "Deinhandy Facebook Organic": "dh_organic_facebook",
    "sparhandy_DE_Alwayson_Podcast": "sh_podcast",
    "simyo_DE_Alwayson_E-Mail": "simyo_e_mail",
    "Deinhandy Affiliate Tracking": "dh_affiliate",
    np.nan: "unknown"
    })
df2[["campaign", "channel"]].value_counts()

,,count
campaign,channel,
DART Search,paid_search,6215305
Sparhandy_DE_Alwayson_DV360_Display_2024,sh_display,1414895
Deinhandy_DE_Alwayson_Affiliate,dh_affiliate,1110327
Sparhandy_DE_Alwayson_Affiliate,sh_affiliate,949003
Sparhandy_DE_Alwayson_Google_DemandGen,Sparhandy_DE_Alwayson_Google_DemandGen,299198
Sparhandy_DE_Alwayson_Facebook_Organic,sh_organic_facebook,253595
sparsim_DE_Alwayson_DV360_Display,sparsim_display,182228
High_DE_Alwayson_DV360_Display_2024,high_display,154215
simyo_DE_Alwayson_DV360_Display,simyo_display,150710


In [10]:
# Show columns, rows, and data types
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11174533 entries, 0 to 11174532
Data columns (total 8 columns):
 #   Column              Dtype         
---  ------              -----         
 0   activity_time       datetime64[ns]
 1   activity            object        
 2   interaction_time    datetime64[ns]
 3   interaction_number  int64         
 4   campaign            object        
 5   interaction_type    object        
 6   ga_cid              object        
 7   channel             object        
dtypes: datetime64[ns](2), int64(1), object(5)
memory usage: 682.0+ MB


In [11]:
# Check for missing values
df2.isnull().sum()

,0
activity_time,0
activity,0
interaction_time,0
interaction_number,0
campaign,0
interaction_type,0
ga_cid,2764495
channel,0


In [12]:
# Filter rows where 'ga_cid' is null
df_nan_cid = df2[df2["ga_cid"].isna()]
df_nan_cid.head()

,activity_time,activity,interaction_time,interaction_number,campaign,interaction_type,ga_cid,channel
8,2026-02-11 20:52:58,Pageview_Server,2026-02-11 20:49:03,2,DART Search,Click,NaN,paid_search
10,2026-02-15 08:35:55,Pageview_Server,2026-02-15 08:28:54,1,DART Search,Click,NaN,paid_search
11,2026-02-12 14:35:12,Pageview_Server,2026-02-12 14:32:27,3,DART Search,Click,NaN,paid_search
12,2026-02-20 15:36:07,Pageview_Server,2026-02-20 14:27:50,3,DART Search,Click,NaN,paid_search
13,2026-02-10 22:52:18,Pageview_Server,2026-02-10 09:12:37,3,DART Search,Click,NaN,paid_search


In [13]:
# Check shape of null 'ga_cid' rows
df_nan_cid.shape

(2764495, 8)

In [14]:
# Check shape of df2
df2.shape

(11174533, 8)

In [15]:
# Drop rows with missing 'ga_cid' values
df3 = df2[df2["ga_cid"].notna()].reset_index(drop=True)
df3.shape

(8410038, 8)

In [16]:
# Check for duplicates
df3.duplicated().sum()

np.int64(7500659)

In [17]:
# Overview of duplicate rows
print(df3.duplicated(keep=False).sum(), "duplizierte Zeilen")
print(df3.duplicated().sum(), "zu entfernende Zeilen")

8362561 duplizierte Zeilen
7500659 zu entfernende Zeilen


In [18]:
# Drop duplicates
df4 = df3.drop_duplicates(keep="first")
df4.shape

(909379, 8)

In [19]:
# Define target variable (sale / order)
order_users       = set(df4[df4["activity"] == "Order_Server"]["ga_cid"])
server_sale_users = set(df4[df4["activity"] == "Server_Sale"]["ga_cid"])

overlap = order_users & server_sale_users

print(f"Order_Server Users:     {len(order_users)}")
print(f"Server_Sale Users:      {len(server_sale_users)}")
print(f"User in beiden:         {len(overlap)}")
print(f"Anteil an Server_Sale in Overlap:  {len(overlap) / len(server_sale_users):.2%}")
print(f"Anteil Order_Server in Overlap: {len(overlap) / len(order_users):.2%}")

order_times = df4[df4["activity"] == "Order_Server"].groupby("ga_cid")["activity_time"].min()
sale_times  = df4[df4["activity"] == "Server_Sale"].groupby("ga_cid")["activity_time"].min()

comparison = pd.DataFrame({
    "order_time": order_times,
    "sale_time":  sale_times
}).dropna()

comparison["order_before_sale"] = comparison["order_time"] < comparison["sale_time"]
print(f"\nOrder vor Sale:  {comparison['order_before_sale'].mean():.2%}")
print(f"Sale vor Order:  {(~comparison['order_before_sale']).mean():.2%}")

Order_Server Users:     1357
Server_Sale Users:      525
User in beiden:         475
Anteil an Server_Sale in Overlap:  90.48%
Anteil Order_Server in Overlap: 35.00%

Order vor Sale:  99.79%
Sale vor Order:  0.21%


### Target variable logic:

*   Order AND Server_Sale -> Server_Sale
*   Only Order_Server    -> Order_Server
*   Only Server_Sale     -> Server_Sale
*   Neither Order/Sale   -> 0

In [20]:
# Verify sale/order
conversion_users = server_sale_users | order_users

print(f"Server_Sale Users: {len(server_sale_users)}")
print(f"Order_Server Users: {len(order_users)}")
print(f"Conversion Users: {len(conversion_users)}")

Server_Sale Users: 525
Order_Server Users: 1357
Conversion Users: 1407


In [21]:
# Check order and timestamp
conversion_tp = (
    df4[df4['activity'].isin(['Order_Server', 'Server_Sale'])]
    .groupby('ga_cid')['interaction_time']
    .apply(set)
    .to_dict()
)

print(f"User mit Conversion-Timestamps: {len(conversion_tp)}")

User mit Conversion-Timestamps: 1407


In [22]:
# Display first 5 rows of df4
df4.head()

,activity_time,activity,interaction_time,interaction_number,campaign,interaction_type,ga_cid,channel
0,2026-02-11 17:55:06,ProductView_Server,2026-02-11 17:43:10,3,DART Search,Click,GA1.1.1833507673.1768199735,paid_search
1,2026-02-20 10:11:46,Pageview_Server,2026-02-20 10:11:42,1,DART Search,Click,GA1.1.1682335648.1771578682,paid_search
2,2026-02-10 21:05:52,Pageview_Server,2026-02-10 21:05:28,1,DART Search,Click,GA1.1.870244807.1769543999,paid_search
3,2026-02-17 20:38:49,Pageview_Server,2026-02-17 14:22:36,2,DART Search,Click,GA1.1.239276648.1771334558,paid_search
4,2026-02-16 09:46:32,Pageview_Server,2026-02-16 09:11:59,2,DART Search,Click,GA1.1.825061615.1769627264,paid_search


In [23]:
# Check maximum interaction time
print(df4["interaction_time"].min(), df4["interaction_time"].max())

2026-02-09 00:00:16 2026-02-22 23:59:33


In [24]:
# Feature engineering: time features
df4["interaction_hour"] = df4["interaction_time"].dt.hour
df4["interaction_minute"] = df4["interaction_time"].dt.minute
df4["interaction_weekday"] = df4["interaction_time"].dt.weekday

df4["interaction_hour_sin"] = np.sin(2 * np.pi * df4["interaction_hour"] / 24)
df4["interaction_hour_cos"] = np.cos(2 * np.pi * df4["interaction_hour"] / 24)
df4["interaction_minute_sin"] = np.sin(2 * np.pi * df4["interaction_minute"] / 60)
df4["interaction_minute_cos"] = np.cos(2 * np.pi * df4["interaction_minute"] / 60)
df4["interaction_weekday_sin"] = np.sin(2 * np.pi * df4["interaction_weekday"] / 7)
df4["interaction_weekday_cos"] = np.cos(2 * np.pi * df4["interaction_weekday"] / 7)

In [25]:
# Verify time features
df4.head()

,activity_time,activity,interaction_time,interaction_number,campaign,interaction_type,ga_cid,channel,interaction_hour,interaction_minute,interaction_weekday,interaction_hour_sin,interaction_hour_cos,interaction_minute_sin,interaction_minute_cos,interaction_weekday_sin,interaction_weekday_cos
0,2026-02-11 17:55:06,ProductView_Server,2026-02-11 17:43:10,3,DART Search,Click,GA1.1.1833507673.1768199735,paid_search,17,43,2,-0.965926,-0.258819,-0.978148,-0.207912,0.974928,-0.222521
1,2026-02-20 10:11:46,Pageview_Server,2026-02-20 10:11:42,1,DART Search,Click,GA1.1.1682335648.1771578682,paid_search,10,11,4,0.500000,-0.866025,0.913545,0.406737,-0.433884,-0.900969
2,2026-02-10 21:05:52,Pageview_Server,2026-02-10 21:05:28,1,DART Search,Click,GA1.1.870244807.1769543999,paid_search,21,5,1,-0.707107,0.707107,0.500000,0.866025,0.781831,0.623490
3,2026-02-17 20:38:49,Pageview_Server,2026-02-17 14:22:36,2,DART Search,Click,GA1.1.239276648.1771334558,paid_search,14,22,1,-0.500000,-0.866025,0.743145,-0.669131,0.781831,0.623490
4,2026-02-16 09:46:32,Pageview_Server,2026-02-16 09:11:59,2,DART Search,Click,GA1.1.825061615.1769627264,paid_search,9,11,0,0.707107,-0.707107,0.913545,0.406737,0.000000,1.000000


In [26]:
# Time since first touchpoint per user (in minutes)
df4["tp_time_delta_min"] = df4.groupby("ga_cid")["interaction_time"].transform(
    lambda x: (x - x.min()).dt.total_seconds() / 60
)

In [27]:
# Create binary flag (1.0 for click, 0.0 for other interactions)
df4["is_click"] = (
    df4["interaction_type"] == "Click"
).astype(float)

print(f"Click-Rate: {df4['is_click'].mean():.1%}")

Click-Rate: 87.0%


In [28]:
# Sort df4 by "ga_cid" and "interaction_time"
df4 = df4.sort_values(["ga_cid", "interaction_time"])
df4.head()

,activity_time,activity,interaction_time,interaction_number,campaign,interaction_type,ga_cid,channel,interaction_hour,interaction_minute,interaction_weekday,interaction_hour_sin,interaction_hour_cos,interaction_minute_sin,interaction_minute_cos,interaction_weekday_sin,interaction_weekday_cos,tp_time_delta_min,is_click
3484186,2026-02-19 11:28:45,ProductView_Server,2026-02-19 11:25:41,1,Sparhandy_DE_Alwayson_Affiliate,Click,GA1.1.1000041479.1771496747,sh_affiliate,11,25,3,0.258819,-0.965926,0.500000,-0.866025,0.433884,-0.900969,0.0,1.0
5484304,2026-02-19 11:28:37,ProductView_Server,2026-02-19 11:25:41,1,Sparhandy_DE_Alwayson_Affiliate,Click,GA1.1.1000041479.1771496747,sh_affiliate,11,25,3,0.258819,-0.965926,0.500000,-0.866025,0.433884,-0.900969,0.0,1.0
536942,2026-02-20 15:27:12,Pageview_Server,2026-02-20 15:26:56,1,DART Search,Click,GA1.1.1000106037.1771597619,paid_search,15,26,4,-0.707107,-0.707107,0.406737,-0.913545,-0.433884,-0.900969,0.0,1.0
649225,2026-02-20 15:27:45,Pageview_Server,2026-02-20 15:26:56,1,DART Search,Click,GA1.1.1000106037.1771597619,paid_search,15,26,4,-0.707107,-0.707107,0.406737,-0.913545,-0.433884,-0.900969,0.0,1.0
900903,2026-02-20 15:27:40,Pageview_Server,2026-02-20 15:26:56,1,DART Search,Click,GA1.1.1000106037.1771597619,paid_search,15,26,4,-0.707107,-0.707107,0.406737,-0.913545,-0.433884,-0.900969,0.0,1.0


In [29]:
# Display columns
df4.columns

Index(['activity_time', 'activity', 'interaction_time', 'interaction_number',
       'campaign', 'interaction_type', 'ga_cid', 'channel', 'interaction_hour',
       'interaction_minute', 'interaction_weekday', 'interaction_hour_sin',
       'interaction_hour_cos', 'interaction_minute_sin',
       'interaction_minute_cos', 'interaction_weekday_sin',
       'interaction_weekday_cos', 'tp_time_delta_min', 'is_click'],
      dtype='object')

In [30]:
# Check unique touchpoints
print(f"Zeilen df4: {len(df4)}")
print(f"Unique Touchpoints: {df4.groupby(['ga_cid', 'interaction_time', 'channel']).ngroups}")

Zeilen df4: 909379
Unique Touchpoints: 86062


In [31]:
# Deduplicate at touchpoint level
df4_dedup = df4.drop_duplicates(
    subset=['ga_cid', 'interaction_time', 'channel'],
    keep='first'
).sort_values(['ga_cid', 'interaction_time'])

print(f"Vorher:  {len(df4)} Zeilen")
print(f"Nachher: {len(df4_dedup)} Zeilen")
print(f"Unique Touchpoints: {df4_dedup.groupby(['ga_cid', 'interaction_time', 'channel']).ngroups}")

Vorher:  909379 Zeilen
Nachher: 86062 Zeilen
Unique Touchpoints: 86062


In [32]:
# Define features
time_feature_cols = [
    # Interaction time features
    "interaction_hour_sin", "interaction_hour_cos",
    "interaction_weekday_sin", "interaction_weekday_cos",
    # Journey features
    "tp_time_delta_min",
    # Interaction type
    "is_click"
]

# Derived feature groups
sincos_cols   = [k for k in time_feature_cols if '_sin' in k or '_cos' in k]
journey_cols  = ['tp_time_delta_min']
binary_cols = ["is_click"]

print(f"Anzahl time_feature_cols: {len(time_feature_cols)}")
print(f"Sin/Cos Features:         {len(sincos_cols)}")
print(f"Journey Features:         {len(journey_cols)}")
print(f"Binary Features:          {len(binary_cols)}")

Anzahl time_feature_cols: 6
Sin/Cos Features:         4
Journey Features:         1
Binary Features:          1


In [33]:
# Sequence building
sequences = []

for user, group in df4_dedup.groupby("ga_cid"):

    group = group.sort_values("interaction_time")

    seq = []
    seq_times = []
    seq_start_time = None

    user_conv_tps = conversion_tp.get(user, set())

    for _, row in group.iterrows():

        # Start time of the current journey
        if seq_start_time is None:
            seq_start_time = row["interaction_time"]

        # Time since first touchpoint of the current journey
        tp_time_delta_min = (
            row["interaction_time"] - seq_start_time
        ).total_seconds() / 60

        # Channel sequence
        seq.append(row["channel"])

        # Touchpoint-level features
        tp_features = {
            col: row[col]
            for col in time_feature_cols
            if col != "tp_time_delta_min"
        }

        # Add journey-specific time feature
        tp_features["tp_time_delta_min"] = tp_time_delta_min

        seq_times.append(tp_features)

        # Conversion reached -> close journey
        if row["interaction_time"] in user_conv_tps:

            sequences.append({
                "ga_cid": user,
                "sequence": seq.copy(),
                "seq_times": seq_times.copy(),
                "touchpoint_sum": len(seq),
                "order": 1
            })

            # Start a new journey after the conversion
            seq = []
            seq_times = []
            seq_start_time = None

    # Remaining journey without conversion
    if seq:
        sequences.append({
            "ga_cid": user,
            "sequence": seq.copy(),
            "seq_times": seq_times.copy(),
            "touchpoint_sum": len(seq),
            "order": 0
        })

df5 = pd.DataFrame(sequences)

In [34]:
# Sicherheitschecks
all_channels = [tp for seq in df5["sequence"] for tp in seq]
unique_channels = sorted(set(all_channels))

# Click-Rate in seq_times prüfen
click_rate_seq = df5['seq_times'].apply( lambda times: np.mean([t['is_click'] for t in times]) ).mean()
print("Shape df5:", df5.shape)
print(f"Conversion Rate: {df5['order'].mean():.2%}")
print(f"Click-Rate (Ø): {click_rate_seq:.1%}")
print(f"Längen stimmen überein: {df5.apply(lambda r: len(r['sequence']) == len(r['seq_times']), axis=1).all()}")
print(f"Max Sequenzlänge: {df5['touchpoint_sum'].max()}")
print(f"Mittlere Sequenzlänge: {df5['touchpoint_sum'].mean():.2f}")
print(f"Unique Channels: {len(unique_channels)}")
print(f"Channels: {unique_channels}")
print(df5['touchpoint_sum'].describe())

Shape df5: (46701, 5)
Conversion Rate: 7.13%
Click-Rate (Ø): 90.9%
Längen stimmen überein: True
Max Sequenzlänge: 150
Mittlere Sequenzlänge: 1.84
Unique Channels: 17
Channels: ['Sparhandy_DE_Alwayson_Google_DemandGen', 'dh_affiliate', 'dh_display', 'dh_e_mail', 'dh_organic_facebook', 'dh_paid_facebook', 'high_display', 'high_paid_facebook', 'paid_search', 'sh_affiliate', 'sh_display', 'sh_e_mail', 'sh_organic_facebook', 'sh_paid_facebook', 'simyo_affiliate', 'simyo_display', 'sparsim_display']
count    46701.000000
mean         1.842830
std          2.268537
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        150.000000
Name: touchpoint_sum, dtype: float64


In [35]:
PROCESSED_DATA_PATH = userdata.get("MTA_PROCESSED_PATH")

df5.to_parquet(
    PROCESSED_DATA_PATH,
    index=False
)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.
